# H&M M5-C 전이 점검 — 개발분할 seed 43

- 최종 마지막 주 test를 열지 않습니다. 기존 개발주(2020-09-02~08)만 사용합니다.
- 동일한 M5-C 수식과 강도를 그대로 옮기며 M1·M3는 정확히 일치하는 완료 곡선만 재사용합니다.
- 신규 학습은 관측 CLV M5-C와 degree층 공동순열 M5-C 두 번뿐입니다.
- 100·200·300 epoch를 기록하지만 판정점은 사전 고정한 300 epoch입니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임(L4 권장)을 연결하세요.'
SOURCE_COMMIT = '58ff14e'
REPO = Path('/content/clv-m5c-hm-screen-' + SOURCE_COMMIT)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', torch.cuda.get_device_name(0), '| code:', SOURCE_COMMIT)


In [ ]:
import clv_m5_user_centered_hm2y_screen as screen
cfg = screen.configure()
print(json.dumps(screen.preflight_summary(cfg), ensure_ascii=False, indent=2))
prepared = screen.prepare(cfg)
assert set(prepared['data']['splits']) == {'val'}
assert cfg.eval_test is False and cfg.eval_holdout is False
assert set(prepared['reference_curve'].model_id) == {screen.M1, screen.M3}
print('사전점검 완료 — 최종 test는 만들지 않았고, 이제 두 M5-C arm만 학습합니다.')


In [ ]:
result = screen.run(cfg, prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


In [ ]:
from IPython.display import display
metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10']
display(result['absolute'][['model_id','seed','epoch',*metrics]])
display(result['comparison'][result['comparison'].metric.isin(metrics)])


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
output = Path('/content/clv_m5_user_centered_hm2y_screen_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
    archive.write(prepared['run_dir'] / 'protocol.json', 'protocol.json')
files.download(str(output))
